# 02 · Discretization and the Green's Matrix

*Turn a continuous slip field into a finite linear system.*

**Learning objectives**

- interpret discretization as a piecewise-constant basis expansion
- build and read the rows and columns of $G$
- use patch-order helpers instead of memorized reshapes
- predict how refinement affects conditioning

**Prerequisites:** Chapter 01; matrix multiplication and least-squares vocabulary  
**Estimated time:** 60–90 minutes

Notation follows the [glossary](../docs/glossary.md); axes, units, signs, and
ordering follow [GeoDef conventions](../docs/conventions.md).


## Motivation

A real fault supports a continuous slip field, but a computer solves for a
finite set of numbers. Discretization chooses the basis in which that field is
approximated. Finer patches can represent shorter wavelengths, but they do not
create information in the data; they create more unknowns that may be nearly
indistinguishable at the surface.


## Theory deepening: basis functions and conditioning

Write continuous slip as a finite expansion

$$s(\xi,\eta)\approx\sum_{j=1}^{N}m_j\,\phi_j(\xi,\eta),$$

where each patch basis $\phi_j$ is one on its patch and zero elsewhere.
Linearity makes column $j$ of $G$ the observation produced by unit slip in
$\phi_j$. Displacement divided by slip gives dimensionless matrix entries
(m/m); projection into LOS changes direction, not units.

Refinement reduces representation error in the source but often increases
$\kappa(G)$ because adjacent or deep columns become similar. Condition number
therefore measures numerical distinguishability, not geological complexity.
Use `reshape_patches` and `flatten_patches` to move between one-dimensional
patch order and the structured grid.


## The design matrix idea

Any *linear* model can be written $\mathbf{d} = G\mathbf{m}$, where each column
of $G$ says how one model parameter contributes to the data. For this reason $G$
is often called the **design matrix**.

The simplest example is fitting a straight line $y = a x + b$. Stacking all
observations,

$$ \mathbf{y} = G\mathbf{m},\qquad
G = \begin{bmatrix} x_1 & 1\\ x_2 & 1\\ \vdots & \vdots\\ x_M & 1\end{bmatrix},\qquad
\mathbf{m} = \begin{bmatrix} a\\ b\end{bmatrix}. $$

Column 1 (the $x_i$) scales with the slope; column 2 (all ones) scales with the
intercept. Fault-slip modeling is the *same idea* — only the columns are elastic
displacement fields instead of `x` and `1`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(7)

# warm-up: the simplest design matrix (line fit)
x = np.linspace(0, 10, 30)
y = 1.8 * x - 0.7 + rng.normal(0, 0.8, x.size)

G_line = np.column_stack([x, np.ones_like(x)])     # (M, 2) design matrix
m_line, *_ = np.linalg.lstsq(G_line, y, rcond=None)
print(f"Recovered slope = {m_line[0]:.2f}, intercept = {m_line[1]:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, s=18, label="data")
ax.plot(x, G_line @ m_line, "r", label="least-squares fit")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend()
ax.set_title("A two-column design matrix")
plt.show()

## From lines to faults

Now replace the two columns of the line-fit example with *hundreds* of columns —
one per fault-patch slip component. Each column is the **surface displacement
field produced by one unit (1 m) of slip on a single patch**, holding every
other patch at zero. Scaled by the actual slip amounts and summed, these basis
fields reproduce the full deformation:

$$ \mathbf{d} = \sum_{k} m_k\, \underbrace{G_{:,k}}_{\text{patch } k \text{ unit response}} = G\mathbf{m}. $$

Let's build a small fault so the matrix stays easy to inspect.

In [ ]:
fault = geodef.Fault.planar(
    lat=0.0, lon=100.0, depth=10e3,
    strike=315.0, dip=20.0,
    length=40e3, width=16e3,
    n_length=8, n_width=4,            # 32 patches -> small, inspectable G
)
N = fault.n_patches
nL, nW = fault.grid_shape

# surface station grid
ox, oy = np.meshgrid(np.linspace(99.6, 100.4, 9), np.linspace(-0.4, 0.4, 9))
obs_lon, obs_lat = ox.ravel(), oy.ravel()
M = obs_lat.size
print(f"{N} patches ({nL} x {nW}),  {M} stations  ->  G is ({3 * M}, {2 * N})")

## Build $G$, column by column

### By hand

Column $k$ of $G$ is just the forward model run with **unit slip on patch $k$
only**. We loop over patches, switch on one unit of slip at a time, and drop the
resulting displacement field into the matching column. The first $N$ columns are
strike-slip; the next $N$ are dip-slip.

In [ ]:
G_manual = np.zeros((3 * M, 2 * N))

for k in range(N):                       # strike-slip columns  [0 : N]
    unit = np.zeros(N); unit[k] = 1.0
    ue, un, uz = fault.displacement(obs_lat, obs_lon,
                                    slip_strike=unit, slip_dip=0.0)
    G_manual[0::3, k], G_manual[1::3, k], G_manual[2::3, k] = ue, un, uz

for k in range(N):                       # dip-slip columns  [N : 2N]
    unit = np.zeros(N); unit[k] = 1.0
    ue, un, uz = fault.displacement(obs_lat, obs_lon,
                                    slip_strike=0.0, slip_dip=unit)
    G_manual[0::3, N + k] = ue
    G_manual[1::3, N + k] = un
    G_manual[2::3, N + k] = uz

print("Built G_manual:", G_manual.shape)

### With geodef

`fault.greens_matrix()` assembles the very same matrix in a single vectorized
call — no Python loop over patches. When you build $G$ for a *dataset*,
`geodef.greens.matrix(fault, dataset)` does this and then projects each column
into the data's observation space. For 3-component GNSS that projection is just
the `[e, n, u]` interleaving, so the two agree exactly.

In [ ]:
G = fault.greens_matrix(obs_lat, obs_lon)
print("greens_matrix == hand-built:", np.allclose(G, G_manual))

# the dataset-aware assembler gives the same matrix for 3-component GNSS
gnss = geodef.data.gnss(lon=obs_lon, lat=obs_lat,
                   east=np.zeros(M), north=np.zeros(M), up=np.zeros(M),
                   sigma_east=np.ones(M), sigma_north=np.ones(M), sigma_up=np.ones(M))
G_proj = geodef.greens.matrix(fault, gnss)
print("greens(fault, gnss) == greens_matrix:", np.allclose(G_proj, G))

### Sidebar: GeoDef caches $G$ for you

Each entry of $G$ comes from evaluating the Okada solution for one patch–station
pair, so assembling $G$ for a large fault and dense data can be slow — and we
usually need the *same* $G$ many times (every step of an inversion, every
regularization weight). `geodef.greens.matrix()` therefore **hashes its inputs
(fault geometry plus observation points) and caches the result to disk**: the
first call computes $G$, and identical later calls just load it. You rarely touch
the cache API directly, but it is there if you need it (`geodef.cache.info()`,
`clear()`, `set_dir()`).

In [ ]:
import time
import tempfile
from geodef import cache

cache.set_dir(tempfile.mkdtemp())          # isolated cache dir just for this demo

t0 = time.perf_counter()
G_a = geodef.greens.matrix(fault, gnss)    # first call: computes G, caches to disk
t1 = time.perf_counter()
G_b = geodef.greens.matrix(fault, gnss)    # second call: loaded from disk
t2 = time.perf_counter()

print(f"first call  (compute + cache): {1e3 * (t1 - t0):6.1f} ms")
print(f"second call (load from cache): {1e3 * (t2 - t1):6.1f} ms")
print("identical result:", np.allclose(G_a, G_b))
print("cache now holds:", geodef.cache.info())

## The structure of $G$

- **Columns are blocked:** `G[:, :N]` are the strike-slip responses, `G[:, N:]`
  the dip-slip responses.
- **Rows are interleaved by station:** `[e₁, n₁, u₁, e₂, n₂, u₂, …]`.

Visualizing the whole matrix makes the two column blocks obvious. The vertical
line separates strike-slip (left) from dip-slip (right).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
vmax = np.abs(G).max()
im = ax.imshow(G, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.axvline(N - 0.5, color="k", lw=1.2)
ax.set_xlabel("model parameter (column):  strike-slip  |  dip-slip")
ax.set_ylabel("observation (row):  [e, n, u] per station")
ax.set_title("The Green's matrix G")
fig.colorbar(im, ax=ax, label="displacement per unit slip (m / m)")
plt.show()

### One column is one patch's "footprint"

Reading a single column back out as a displacement field shows what it really
is: the surface motion from 1 m of slip on one patch. Here is the dip-slip
column for a patch in the middle of the fault, drawn over the highlighted patch.

In [ ]:
k = fault.patch_index(strike_idx=4, dip_idx=1)
col = G[:, N + k]                        # dip-slip column for patch k
ce, cn, cu = col[0::3], col[1::3], col[2::3]

resp = geodef.data.gnss(lon=obs_lon, lat=obs_lat, east=ce, north=cn, up=cu,
                   sigma_east=np.ones(M) * 1e-6, sigma_north=np.ones(M) * 1e-6, sigma_up=np.ones(M) * 1e-6)

fig, ax = plt.subplots(figsize=(6.5, 6))
highlight = np.zeros(N); highlight[k] = 1.0
geodef.plot.slip(fault, highlight, ax=ax, cmap="Reds", colorbar=False,
                 coords="geographic")

scale = 12.0 / np.hypot(ce, cn).max()
geodef.plot.vectors(resp, fault, ax=ax, components="horizontal", scale=scale,
                    title=f"Column {N + k}: unit dip-slip on patch {k}")
plt.show()

## Forward prediction with many patches

With $G$ in hand, predicting data for *any* slip model is one matrix multiply.
Let's build a slip distribution with two asperities and predict the surface
displacements — note how little code this takes now that the physics lives in
$G$.

In [ ]:
i = np.arange(N) % nL
j = np.arange(N) // nL
slip = (1.0 * np.exp(-(((i - 2) / 1.2) ** 2 + ((j - 1) / 1.0) ** 2))
        + 0.7 * np.exp(-(((i - 6) / 1.0) ** 2 + ((j - 2) / 0.8) ** 2)))

m = np.concatenate([np.zeros(N), slip])      # dip-slip only
d = G @ m
pe, pn, pu = d[0::3], d[1::3], d[2::3]

pred = geodef.data.gnss(lon=obs_lon, lat=obs_lat, east=pe, north=pn, up=pu,
                   sigma_east=np.ones(M) * 1e-3, sigma_north=np.ones(M) * 1e-3, sigma_up=np.ones(M) * 1e-3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
geodef.plot.slip(fault, slip, ax=axes[0], cmap="YlOrRd",
                 coords="geographic",
                 colorbar_label="Dip-slip (m)",
                 title="Slip model (two asperities)")
scale = 12.0 / np.hypot(pe, pn).max()
geodef.plot.vectors(pred, fault, ax=axes[1], components="both", scale=scale,
                    title="Predicted displacements")
plt.show()

## Checkpoint questions

**What physical object is one column of $G$?**

<details><summary>Answer</summary>



The projected observation field from unit slip on one patch and component.



</details>

**Does refining patches improve resolution?**

<details><summary>Answer</summary>



It improves representation capacity; resolution improves only if the data distinguish the added columns.



</details>

**Why use cache keys that include precision and medium?**

<details><summary>Answer</summary>



Either changes the computed matrix, so omitting it could silently return the wrong result.



</details>


## Common mistakes

- Assuming `reshape(NW, NL)` without checking patch order can transpose or permute slip.
- Comparing condition numbers of differently scaled matrices can confuse units with information.
- Treating a cached matrix as trustworthy without complete keys risks stale physics.


## Recap

- Discretization is a basis choice, not a resolution claim.
- Columns encode unit patch responses; rows encode measured components.
- Refinement often increases column similarity and ill-conditioning.

Return to the [workflow decision guides](../docs/workflow.md) when adapting
this method. The course map in [the tutorial README](README.md) identifies the
next chapter and optional branches.


## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are published separately in `solutions/02_discretization_and_g_matrix_solutions.ipynb`.

1. Refine each grid dimension and tabulate shape and condition number.
2. Compare strike- and dip-slip columns for the same patch.
3. Compare adjacent shallow and deep columns with normalized dot products.
4. Challenge: construct rows for an East-only instrument and verify them against raw ENU rows.


## Further reading

- Aster, Borchers & Thurber (2018), discretization and inverse problems.
- Menke (2018), design matrices and conditioning.
- Okada (1985), the rectangular kernel behind each column.
